# Compute sidewalk width
Using centerline extraction (also called skeleton line, axis line, or medial line extraction)

In [1]:
# Add project src to path.
import set_path

import numpy as np
import os
import pathlib
import pickle
import pandas as pd
import geopandas as gpd
from tqdm.notebook import tqdm_notebook
tqdm_notebook.pandas()
import shapely.ops as so
import shapely.geometry as sg

import upcp.utils.bgt_utils as bgt_utils
import upcp.utils.las_utils as las_utils

import upc_sw.poly_utils as poly_utils

In [2]:
import warnings  # temporary, to supress deprecationwarnings from shapely
warnings.filterwarnings('ignore')

In [ ]:
tile_code = "113300_489300"

# Paths
pc_data_folder = "/home/vishruth/Backup/pointcloud/georizon-dataset-01929985-8de2-7391-a90a-4cdfaa30047a/"
out_folder = '../datasets/output/'  

# Save intermediate output in case of errors
tmp_file = f'{out_folder}sw_seg_tmp.pkl'

# Set Coordinate Reference System
CRS = 'epsg:28992' 

# Whether to merge sidewalks before segmentation and width computation
merge_sidewalks = True

# Tolerance for centerline simplification
simplify_tolerance = 0.2

# Min area size of a sidewalk polygon in sqm for which width will be computed
min_area_size = 5

# Minimum length for short-ends (in meters), otherwise removed
min_se_length = 5

# Max segment length in meters
max_seg_length = 2 

# Resolution (in m) for min and avg width computation
width_resolution = 1

# Precision (in decimals) for min and avg width computation
width_precision = 1

# Create folder if it doesn't exist
pathlib.Path(out_folder).mkdir(parents=True, exist_ok=True)

## Read the sidewalk and obstacle data

In [4]:
# Read sidewalk with obstacle data
obstacle_file = f'{out_folder}/sidewalks_with_obstacles_{tile_code}.gpkg'
df = gpd.read_file(obstacle_file, geometry='geometry', crs=CRS)

if merge_sidewalks:
    # Merge sidewalk polygons
    df = gpd.GeoDataFrame(geometry=gpd.GeoSeries([geom for geom in df.unary_union.geoms]), crs=CRS)
    df['ogc_fid'] = range(0, len(df))  
    
else:
    # Explode MultiPolygons into their parts
    df = df.explode(index_parts=False)

# Ignore sidewalk polygons that are too small
df = df[df.area > min_area_size]

## Calculate width along centerline segments

In [5]:
def get_points_on_line(line, distance_delta):  
    # Generate equidistant points
    distances = np.arange(0, line.length, distance_delta)
    points = sg.MultiPoint([line.interpolate(distance) for distance in distances])
    return points

def split_line_by_points(line, points, tolerance: float=0.001):
    return so.split(so.snap(line, points, tolerance), points)

In [6]:
def get_segments_width_cut(row, max_seg_length):
    # Get centerlines.
    cl = poly_utils.get_centerlines(row.geometry)
    # Merge linestrings.
    cl = so.linemerge(cl)
    # Remove short line ends and dead-ends.
    cl = poly_utils.remove_short_lines(cl, min_se_length)
    # Simplify lines.
    cl = cl.simplify(simplify_tolerance, preserve_topology=True)
    # Segment lines 
    segments_long = poly_utils.get_segments(cl)   
    # Cut segments (with maximum segment length)
    segments = []
    for seg in segments_long:
        points_on_line = get_points_on_line(seg, max_seg_length)
        seg_cut = split_line_by_points(seg, points_on_line)
        segments.extend(seg_cut)
    # Compute avg and min width per cut segment   
    avg_width, min_width = poly_utils.get_avg_width(
                    row.geometry, segments, width_resolution, width_precision)
    return {'segments_long': segments_long, 'segments': segments, 
            'avg_width': avg_width, 'min_width': min_width, 'sidewalk_id': row.ogc_fid}      

In [7]:
from shapely.ops import linemerge

In [8]:
from shapely.geometry import LineString, MultiLineString, GeometryCollection

def ensure_lines(geometry):
    if isinstance(geometry, GeometryCollection):
        return [g for g in geometry.geoms if isinstance(g, LineString)]
    elif isinstance(geometry, LineString):
        return [geometry]
    elif isinstance(geometry, MultiLineString):
        return list(geometry.geoms)
    else:
        return []

In [9]:
def get_segments_width_cut_check(row, max_seg_length):
    # Get centerlines.
    cl = poly_utils.get_centerlines(row.geometry)
    # Merge linestrings.
    cl = linemerge(cl.geometry)

    # Remove short line ends and dead-ends.
    cl = poly_utils.remove_short_lines(cl, min_se_length)
    # Simplify lines.
    cl = cl.simplify(simplify_tolerance, preserve_topology=True)
    # Segment lines 
    segments_long = poly_utils.get_segments(cl)   
    # Cut segments (with maximum segment length)
    segments = []
    for seg in segments_long:
        points_on_line = get_points_on_line(seg, max_seg_length)
        seg_cut = split_line_by_points(seg, points_on_line)
        segments.extend(ensure_lines(seg_cut))
    # Compute avg and min width per cut segment   
    avg_width, min_width = poly_utils.get_avg_width(
                    row.geometry, segments, width_resolution, width_precision)
    return {'segments_long': segments_long, 'segments': segments, 
            'avg_width': avg_width, 'min_width': min_width, 'sidewalk_id': row.ogc_fid}      

In [10]:
# if you get an error here, make sure you use tqdm>=4.61.2
segment_df = pd.DataFrame(df.progress_apply(get_segments_width_cut_check, 
                                            max_seg_length=max_seg_length,                                                     
                                            axis=1).values.tolist())

with open(tmp_file, 'wb') as f:
    pickle.dump(segment_df.to_dict(), f)

  0%|          | 0/3 [00:00<?, ?it/s]

'Polygon' object is not iterable


AttributeError: 'float' object has no attribute 'geometry'

### Explode into individual segments

In [ ]:
segment_df = pd.concat([gpd.GeoDataFrame({'geometry': row.segments,
                                          'avg_width': row.avg_width,
                                          'min_width': row.min_width,
                                          'sidewalk_id': row.sidewalk_id} 
                                        )
                         for _, row in segment_df.iterrows()],
                       ignore_index=True)
segment_df.set_crs(crs=CRS, inplace=True);

with open(tmp_file, 'wb') as f:
    pickle.dump(segment_df.to_dict(), f)

## Check coverage of point cloud data on sidewalks

In [ ]:
pc_file_prefix = 'processed'

# Get a list of all tilecodes for which we have two runs.
all_tiles = (las_utils.get_tilecodes_from_folder(f'{pc_data_folder}run1/', las_prefix=pc_file_prefix)
             .intersection(las_utils.get_tilecodes_from_folder(f'{pc_data_folder}run2/', las_prefix=pc_file_prefix)))
all_tiles_poly = so.unary_union([poly_utils.tilecode_to_poly(tile) for tile in all_tiles])

segment_df['pc_coverage'] = segment_df.intersects(all_tiles_poly)

In [ ]:
segment_df['pc_coverage'].value_counts()

pc_coverage
False    381
Name: count, dtype: int64

## Store output

In [ ]:
segments_file = f'{out_folder}sidewalk_segments_{tile_code}.gpkg'
segment_df.to_file(segments_file, driver='GPKG')

# Delete intermediate output
if os.path.exists(tmp_file):
    os.remove(tmp_file)

## Plot results

In [ ]:
# Optional: read the saved segments file.
segment_df = gpd.read_file(f'{out_folder}sidewalk_segments_{tile_code}.gpkg', crs=CRS)

In [ ]:
segment_df

,avg_width,min_width,sidewalk_id,pc_coverage,geometry
0,1.3,0.3,0,False,"LINESTRING (113301.911 489326.685, 113300.234 ..."
1,0.2,0.2,0,False,"LINESTRING (113300.234 489325.595, 113299.956 ..."
2,0.4,0.2,0,False,"LINESTRING (113299.956 489325.414, 113299.726 ..."
3,0.5,0.5,0,False,"LINESTRING (113299.726 489323.427, 113299.676 ..."
4,0.6,0.5,0,False,"LINESTRING (113299.676 489322.992, 113300.027 ..."
...,...,...,...,...,...
376,1.2,1.2,2,False,"LINESTRING (113363.894 489340.948, 113361.894 ..."
377,1.3,1.2,2,False,"LINESTRING (113361.894 489340.934, 113359.894 ..."
378,1.3,1.3,2,False,"LINESTRING (113359.894 489340.919, 113358.474 ..."
379,1.8,1.7,2,False,"LINESTRING (113369.006 489340.697, 113369.083 ..."
